In [51]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests
from dotenv import load_dotenv
import json

load_dotenv()

True

In [52]:
# tool create

@tool
def get_conversion_factor(base_currency: str, target_currency:str) -> float:
    """This function fetches the currency conversion factor between a given base currency and a target currency."""
    url = f'https://v6.exchangerate-api.com/v6/6846cfefe2e1ab40a86ac8f9/pair/{base_currency}/{target_currency}'

    response = requests.get(url)

    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """This function converts a given amount in the base currency to the target currency using the conversion rate."""
    return base_currency_value * conversion_rate


In [53]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1788825601,
 'time_last_update_utc': 'Tue, 08 Sep 2026 00:00:01 +0000',
 'time_next_update_unix': 1788912001,
 'time_next_update_utc': 'Wed, 09 Sep 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.5485}

In [54]:
convert.invoke({'base_currency_value': 100, 'conversion_rate': 94.5485})

9454.85

In [55]:
# tool binding
llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash')
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [56]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that convert 10 USD to INR.')]
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that convert 10 USD to INR.', additional_kwargs={}, response_metadata={})]

In [57]:
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)
ai_message

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, '__gemini_function_call_thought_signatures__': {'call_216003': 'EtgICtUIARFNMg9hjqQm1nZOPGneAntiUYbIeFZQ/tSHPWaCzrzqU/5xbuHQOHjFXV3VvgY9/+S7RFsG18KKIhn1N9Uzt+MukvDj3B5VyDRU+aE5VVDYH8v/kaCjy7O9NXOvvRD5mw1oL9vonbcDRCq9YHr1v6AisMMQeWMZschEsenmzXz2ohNB2qxR4JAAOAPsQO0VrHwaPMB6UjVdFXNScR1NBgyLUN3ydLSQMGK0zSdkFLW3/xeNO+f+JA9Ri9PoytwohAvCIw+EJy4rdir+Sk/K73rTxv6zFoy+RSfzC38i8Ii4JvS9Of1lE5PSxoVz/BYc5EH7UQRa2ifmgT7ivbQQxAnF2hspwbTCsyZ3aI2phqMumes1MhAxuEVkpdPrgrc5Xh6Cb1e73fqQxfM8NqBmW1WNZMmwib85xanRx6dQMbly/3bS5D+R4YDudlSIomOzGQiFXr9B3KSOyKLAmsGeMMctyU9i0EONtIX2IXVe68A5wztrYyVY525IU/LGfRv8Visq2R/byjxSS8LCc9PZeHOZTtWnrm05RPSpf2xPtEdVrSLQ/iUVJ5mnJUsx4jO7LL5rzERtcRFEPxEHrW81KTpKliVcTJkUbjrL/TTHYIjulBXQevczYGhrGxtbf/c5xzmjtyQ1/5mGAwcDv7odO0Sj/3+JHimrZyfKDkX58UVAxuP/mDTGN/YZYD4m9amtZk/N7Cv+hmuoeqHG2nLRKyBINN+6KBarJZFM+s0OwBVcVnMr+vRva9vQRWLIz57U

In [58]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_216003',
  'type': 'tool_call'}]

In [59]:
for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        # fetch conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to messages list
        messages.append(tool_message1)
    # execute the 2nd tool using the conversion rate
    if tool_call['name'] == 'convert':
        # fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [60]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that convert 10 USD to INR.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, '__gemini_function_call_thought_signatures__': {'call_216003': 'EtgICtUIARFNMg9hjqQm1nZOPGneAntiUYbIeFZQ/tSHPWaCzrzqU/5xbuHQOHjFXV3VvgY9/+S7RFsG18KKIhn1N9Uzt+MukvDj3B5VyDRU+aE5VVDYH8v/kaCjy7O9NXOvvRD5mw1oL9vonbcDRCq9YHr1v6AisMMQeWMZschEsenmzXz2ohNB2qxR4JAAOAPsQO0VrHwaPMB6UjVdFXNScR1NBgyLUN3ydLSQMGK0zSdkFLW3/xeNO+f+JA9Ri9PoytwohAvCIw+EJy4rdir+Sk/K73rTxv6zFoy+RSfzC38i8Ii4JvS9Of1lE5PSxoVz/BYc5EH7UQRa2ifmgT7ivbQQxAnF2hspwbTCsyZ3aI2phqMumes1MhAxuEVkpdPrgrc5Xh6Cb1e73fqQxfM8NqBmW1WNZMmwib85xanRx6dQMbly/3bS5D+R4YDudlSIomOzGQiFXr9B3KSOyKLAmsGeMMctyU9i0EONtIX2IXVe68A5wztrYyVY525IU/LGfRv8Visq2R/byjxSS8LCc9PZeHOZTtWnrm05RPSpf2xPtEdVrSLQ/iUVJ5mnJUsx4jO7LL5rzERtcRFEPxEHrW81KTpKliVcT

In [ ]:
llm_with_tools.invoke(messages).content

[]